# Lab 17-07 · Entropy, Capacity and Water-Filling

Five parts, each built on `infotheory.py`. **Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import math
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import infotheory as it

## Part 1 · Compression

*Predict:* the entropy per character of the sample text (letters and spaces only), and how close a Huffman code on single characters comes to it.

In [2]:
text = it.letters_only(it.SAMPLE_TEXT)
H, probs = it.empirical_entropy(text)
code = it.huffman(probs)
L = it.average_length(code, probs)
bits = it.encode(text, code)
assert "".join(it.decode(bits, code)) == text   # lossless
print(f"{len(text)} characters, {len(probs)} distinct symbols, log2(alphabet) = {math.log2(len(probs)):.2f} bits")
print(f"entropy {H:.3f} bits/char, Huffman {L:.3f} bits/char, Kraft sum {it.kraft([len(w) for w in code.values()]):.3f}")
for s in sorted(code, key=lambda s: -probs[s])[:6]:
    print(f"  {s!r}: p = {probs[s]:.3f}, codeword {code[s]}")
assert abs(H - 4.01) < 0.005 and abs(L - 4.05) < 0.005

700 characters, 25 distinct symbols, log2(alphabet) = 4.64 bits
entropy 4.011 bits/char, Huffman 4.049 bits/char, Kraft sum 1.000
  ' ': p = 0.183, codeword 00
  'e': p = 0.110, codeword 010
  'a': p = 0.080, codeword 1110
  't': p = 0.069, codeword 1100
  'o': p = 0.067, codeword 1010
  'n': p = 0.064, codeword 1001


## Part 2 · Capacity by algorithm

*Predict:* the BSC capacity at $p$ = 0.1, and whether the Z-channel's best input is uniform.

In [3]:
bsc = np.array([[0.9, 0.1], [0.1, 0.9]])
c, p, n = it.blahut_arimoto(bsc)
print(f"BSC p = 0.1: Blahut-Arimoto C = {c:.4f} with input {np.round(p, 4)} ({n} iterations); closed form {it.bsc_capacity(0.1):.4f}")
z = np.array([[1.0, 0.0], [0.5, 0.5]])
cz, pz, nz = it.blahut_arimoto(z)
print(f"Z-channel q = 0.5: C = {cz:.4f}, P(X=1) = {pz[1]:.4f} ({nz} iterations); closed form {it.z_capacity(0.5)}")
assert abs(c - 0.531) < 5e-4 and abs(pz[1] - 0.4) < 1e-4

BSC p = 0.1: Blahut-Arimoto C = 0.5310 with input [0.5 0.5] (1 iterations); closed form 0.5310
Z-channel q = 0.5: C = 0.3219, P(X=1) = 0.4000 (55 iterations); closed form (0.32192809488736235, 0.4)


## Part 3 · Bandwidth at fixed power

*Predict:* the capacity ratio when bandwidth doubles at fixed $S/N_0$, starting from SNR = 30 dB and from −10 dB.

In [4]:
for snr_db in (30, -10):
    sn0 = 10 ** (snr_db / 10) * 1e6          # S/N0 that gives this SNR in 1 MHz
    c1, c2 = it.capacity_from_sn0(1e6, sn0), it.capacity_from_sn0(2e6, sn0)
    print(f"SNR {snr_db:+d} dB in 1 MHz: {c1 / 1e6:.4f} Mb/s; in 2 MHz: {c2 / 1e6:.4f} Mb/s; ratio {c2 / c1:.3f}; limit {sn0 / math.log(2) / 1e6:.4f} Mb/s")
B = np.logspace(4, 9, 200)
sn0 = 1e6
plt.semilogx(B, [it.capacity_from_sn0(b, sn0) / 1e6 for b in B])
plt.axhline(sn0 / math.log(2) / 1e6, ls=":", color="k")
plt.xlabel("bandwidth (Hz)"); plt.ylabel("capacity (Mb/s)"); plt.title("S/N0 = 60 dB-Hz: capacity saturates at 1.443 Mb/s"); plt.show()

SNR +30 dB in 1 MHz: 9.9672 Mb/s; in 2 MHz: 17.9373 Mb/s; ratio 1.800; limit 1442.6950 Mb/s
SNR -10 dB in 1 MHz: 0.1375 Mb/s; in 2 MHz: 0.1408 Mb/s; ratio 1.024; limit 0.1443 Mb/s


## Part 4 · Constrained inputs

*Predict:* the $E_b/N_0$ at which Gaussian-input, BPSK-soft and BPSK-hard capacity each reach rate 1/2.

In [5]:
es = np.linspace(-10, 15, 201)
lin = 10 ** (es / 10)
for fn, name in ((it.gaussian_real_capacity, "Gaussian input"), (it.bpsk_soft_capacity, "BPSK soft"), (it.bpsk_hard_capacity, "BPSK hard")):
    plt.plot(es, [fn(x) for x in lin], label=name)
    x = it.required_es_n0_db(fn, 0.5)
    print(f"{name:15s}: rate 1/2 at Es/N0 = {x:6.2f} dB, Eb/N0 = {x + 10 * math.log10(2):5.2f} dB")
plt.axhline(0.5, ls=":", color="k"); plt.xlabel("Es/N0 (dB)"); plt.ylabel("bits per real use"); plt.legend(); plt.show()
print("Shannon limit at eta = 1, 2, 4, 6 b/s/Hz:", [round(it.shannon_limit_db(e), 2) for e in (1, 2, 4, 6)], "dB; ultimate", round(it.shannon_limit_db(0), 2))

Gaussian input : rate 1/2 at Es/N0 =  -3.01 dB, Eb/N0 = -0.00 dB
BPSK soft      : rate 1/2 at Es/N0 =  -2.82 dB, Eb/N0 =  0.19 dB
BPSK hard      : rate 1/2 at Es/N0 =  -1.24 dB, Eb/N0 =  1.77 dB
Shannon limit at eta = 1, 2, 4, 6 b/s/Hz: [0.0, 1.76, 5.74, 10.21] dB; ultimate -1.59


## Part 5 · Water-filling over 64 subcarriers

*Predict:* which subcarriers of $H_k = 1 + 0.9e^{-j2\pi k/64}$ get no power when the total is 64 (transmit SNR 0 dB per unit power).

In [6]:
k = np.arange(64)
g = np.abs(1 + 0.9 * np.exp(-2j * np.pi * k / 64)) ** 2
p, mu, C = it.waterfill(g, 64)
Ceq = it.equal_power_capacity(g, 64)
off = np.flatnonzero(p == 0)
print(f"water level {mu:.3f}; {len(off)} subcarriers off (k = {off[0]}..{off[-1]})")
print(f"capacity per subcarrier: water-filling {C / 64:.3f}, equal power {Ceq / 64:.3f} b/s/Hz")
fig, ax = plt.subplots()
ax.bar(k, 1 / g, color="0.7", label="floor 1/g_k")
ax.bar(k, p, bottom=1 / g, label="power p_k")
ax.axhline(mu, color="k", ls=":", label="water level")
ax.set_ylim(0, 4); ax.set_xlabel("subcarrier k"); ax.legend(); plt.show()
assert len(off) == 15 and abs(C / 64 - 1.487) < 5e-4

water level 1.899; 15 subcarriers off (k = 25..39)
capacity per subcarrier: water-filling 1.487, equal power 1.313 b/s/Hz
